  #                                                                       # *****DATA PRE PROCESSING*****

# ***Demography***

In [2]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


import warnings
warnings.simplefilter("ignore", UserWarning)

In [108]:
# from pathlib import Path
# data_folder = Path(
#     r"C:\Users\vella\Desktop\Numpy Ninja\Team3_PyCoders_PythonHackathon_SEP2026\Python_Hackathon_Sep_2026\Python_Hackathon_Sep_2026\cardiac_failure")

### **1. Read the CSV file and inspect the data. This confirms the file loaded correctly and shows missing values before you change anything**

In [109]:
# Read the demography CSV File and inspect the data 

dfDEMO = pd.read_csv("data_files/demography.csv")
df_original = dfDEMO.copy()

print("Rows and columns:", dfDEMO.shape)
display(dfDEMO.head())
dfDEMO.info()

print("\nMissing values:")
display(dfDEMO.isna().sum())

print(
    "Duplicate patient IDs:",
    dfDEMO["inpatient_number"].duplicated().sum()
)



Rows and columns: (2009, 7)


,inpatient_number,gender,weight,height,bmi,occupation,agecat
0,5,NaN,NaN,NaN,46.000000,NaN,NaN
1,827040,Female,50.0,1.45,23.781213,NaN,69-79
2,857781,Male,50.0,1.64,18.590125,UrbanResident,69-79
3,743087,Female,51.0,1.63,19.195303,UrbanResident,69-79
4,866418,Male,70.0,1.70,24.221453,farmer,59-69


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   inpatient_number  2009 non-null   int64  
 1   gender            2008 non-null   object 
 2   weight            2008 non-null   float64
 3   height            2008 non-null   float64
 4   bmi               2009 non-null   float64
 5   occupation        1981 non-null   object 
 6   agecat            2008 non-null   object 
dtypes: float64(3), int64(1), object(3)
memory usage: 110.0+ KB

Missing values:


inpatient_number     0
gender               1
weight               1
height               1
bmi                  0
occupation          28
agecat               1
dtype: int64

Duplicate patient IDs: 0


### **2. Removed the empty record. Patient ID 5 has no gender, weight, height,occupation, or age category. Its BMI value alone is not enuogh to use the record for demographic analysis. So this removes one row**


In [110]:
dfDEMO = dfDEMO.dropna(
    subset=["gender", "weight", "height", "occupation", "agecat"],
    how="all"
).copy()

### **3. Renamed column names for best readability and consistency across datasets for less confusion during analysis**

In [111]:
dfDEMO = dfDEMO.rename(columns={
    "inpatient_number": "patient_id",
    "agecat" : "age_category"
})

### **4. Removed extra spaces from the text columns so values such as "Male" and " Male " are treated as the same category. I also make the occupation names consistent and easier to read. This prevents one category from appearing under different labels in charts and counts.**

In [112]:
print(dfDEMO.columns.tolist())

text_columns = ["gender", "occupation", "age_category"]

for column in text_columns:
    dfDEMO[column] = dfDEMO[column].astype("string").str.strip()

dfDEMO["occupation"] = dfDEMO["occupation"].replace({
    "UrbanResident": "Urban Resident",
    "farmer": "Farmer",
    "worker": "Worker"
})

dfDEMO["occupation"] = dfDEMO["occupation"].fillna("Unknown")



['patient_id', 'gender', 'weight', 'height', 'bmi', 'occupation', 'age_category']


### **5. Impossible measurements, Weight values of zero or less and height values below 1.0 are flagged for review. The code marks those rows in measurement and changes the flagged values to missing so they do not affect BMI calculations. It then counts how many records were flagged.***

In [113]:
dfDEMO["measurement"] = pd.Series(False, index=dfDEMO.index)

invalid_weight = dfDEMO["weight"] <= 0
invalid_height = dfDEMO["height"] < 1.0

dfDEMO.loc[invalid_weight | invalid_height, "measurement"] = True

dfDEMO.loc[invalid_weight, "weight"] = np.nan
dfDEMO.loc[invalid_height, "height"] = np.nan

print("Records with measurement issues:", dfDEMO["measurement"].sum())

Records with measurement issues: 7


### **6. Recalculated BMI from Valid measurements. The orginial BMI values match the recorded weights and heights, including the incorrect heights and zero weights, Recalculating after flagging those measurements makes BMI missing for the seven affected records**

In [114]:
dfDEMO["bmi_original"] = dfDEMO["bmi"].round(2)

dfDEMO["bmi"] = dfDEMO["weight"] / (dfDEMO["height"] ** 2)
dfDEMO["bmi"] = dfDEMO["bmi"].round(2)


### **7. Removed Unrealistic BMI values and validated age category consistency**

In [115]:
dfDEMO["bmi"] = pd.to_numeric(dfDEMO["bmi"], errors="coerce")
dfDEMO.loc[(dfDEMO["bmi"] < 15) | (dfDEMO["bmi"] > 60), "bmi"] = np.nan
dfDEMO["age_category"] = dfDEMO["age_category"].astype(str).str.strip()
dfDEMO["age_category"] = dfDEMO["age_category"].str.title()

### **8. Reviewed the cleaned data to make sure it is ready for analysis. I check the number of rows and unique patients, see which values are still missing, review the weight, height, and BMI ranges, and count each category. Finally, I display the records flagged for measurement issues so I can inspect them before using them in calculations.**

In [116]:
print("Rows:", len(dfDEMO))
print("Unique patient IDs:", dfDEMO["patient_id"].nunique())
print("\nMissing values:")
print(dfDEMO.isna().sum())

print("\nNumeric summary:")
print(dfDEMO[["weight", "height", "bmi"]].describe())

print("\nCategory counts:")
for column in ["gender", "occupation", "age_category"]:
    print(f"\n{column}")
    print(dfDEMO[column].value_counts(dropna=False))

print("\nRecords needing measurement review:")
print(
    dfDEMO.loc[
        dfDEMO["measurement"],
        ["patient_id", "weight", "height", "bmi_original"]
    ]
)

Rows: 2008
Unique patient IDs: 2008

Missing values:
patient_id       0
gender           0
weight           3
height           4
bmi             39
occupation       0
age_category     0
measurement      0
bmi_original     0
dtype: int64

Numeric summary:
            weight       height          bmi
count  2005.000000  2004.000000  1969.000000
mean     52.562244     1.570110    21.407984
std      10.713048     0.081954     3.732060
min       8.000000     1.200000    15.070000
25%      45.000000     1.500000    18.670000
50%      50.000000     1.560000    20.810000
75%      60.000000     1.620000    23.440000
max     115.000000     1.830000    39.110000

Category counts:

gender
gender
Female    1163
Male       845
Name: count, dtype: Int64

occupation
occupation
Urban Resident    1670
Farmer             198
Others              89
Unknown             27
Worker              17
Officer              7
Name: count, dtype: Int64

age_category
age_category
69-79     715
79-89     646
59-69    

### **9. Save analysis and reviewing the files**

In [117]:
dfDEMO.to_csv("cleaned_files/demography_clean.csv", index=False)

#                                                          ***Patient History***

### **1. Load the CSV File and check the size. column types, missing values, and patient IDs before changing anything**

In [118]:
dfPH = pd.read_csv("data_files/patienthistory.csv")
df_original = dfPH.copy()

print("Rows and columns:", dfPH.shape)
display(dfPH.head())
dfPH.info()

print("\nMissing values:")
display(dfPH.isna().sum())

print(
    "Duplicate patient IDs:",
    dfPH["inpatient_number"].duplicated().sum()
)


Rows and columns: (2008, 17)


,inpatient_number,cerebrovascular_disease,dementia,chronic_obstructive_pulmonary_disease,connective_tissue_disease,peptic_ulcer_disease,diabetes,moderate_to_severe_chronic_kidney_disease,hemiplegia,leukemia,malignant_lymphoma,solid_tumor,liver_disease,aids,cci_score,type_ii_respiratory_failure,acute_renal_failure
0,857781,0,0,1,0,0.0,1,0.0,0,0,0,0,0.0,0,2.0,nontypeii,0
1,743087,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
2,866418,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
3,775928,0,0,1,0,0.0,0,1.0,0,0,0,0,0.0,0,2.0,nontypeii,0
4,810128,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 17 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   inpatient_number                           2008 non-null   int64  
 1   cerebrovascular_disease                    2008 non-null   int64  
 2   dementia                                   2008 non-null   int64  
 3   chronic_obstructive_pulmonary_disease      2008 non-null   int64  
 4   connective_tissue_disease                  2008 non-null   int64  
 5   peptic_ulcer_disease                       2006 non-null   float64
 6   diabetes                                   2008 non-null   int64  
 7   moderate_to_severe_chronic_kidney_disease  2006 non-null   float64
 8   hemiplegia                                 2008 non-null   int64  
 9   leukemia                                   2008 non-null   int64  
 10  malignant_lymphoma      

inpatient_number                             0
cerebrovascular_disease                      0
dementia                                     0
chronic_obstructive_pulmonary_disease        0
connective_tissue_disease                    0
peptic_ulcer_disease                         2
diabetes                                     0
moderate_to_severe_chronic_kidney_disease    2
hemiplegia                                   0
leukemia                                     0
malignant_lymphoma                           0
solid_tumor                                  0
liver_disease                                1
aids                                         0
cci_score                                    5
type_ii_respiratory_failure                  0
acute_renal_failure                          0
dtype: int64

Duplicate patient IDs: 0


### ***2. Renamed the column name inpatient_number to patient_id***

In [119]:
dfPH = dfPH.rename(columns={
    "inpatient_number": "patient_id",
    "type_ii_respiratory_failure": "respiratory_failure_type2",
    "moderate_to_severe_chronic_kidney_disease": "mod_severe_ckd",
    "chronic_obstructive_pulmonary_disease": "copd"
})

### ***3. Condition should check only 0,1, or missing values reviewing unexpected values before converting Types***

In [120]:
condition_columns = [
    "cerebrovascular_disease",
    "dementia",
    "copd",
    "connective_tissue_disease",
    "peptic_ulcer_disease",
    "diabetes",
    "mod_severe_ckd",
    "hemiplegia",
    "leukemia",
    "malignant_lymphoma",
    "solid_tumor",
    "liver_disease",
    "aids",
    "acute_renal_failure",
]

for column in condition_columns:
    print(f"\n{column}")
    print(dfPH[column].value_counts(dropna=False))


cerebrovascular_disease
cerebrovascular_disease
0    1858
1     150
Name: count, dtype: int64

dementia
dementia
0    1893
1     115
Name: count, dtype: int64

copd
copd
0    1775
1     233
Name: count, dtype: int64

connective_tissue_disease
connective_tissue_disease
0    2004
1       4
Name: count, dtype: int64

peptic_ulcer_disease
peptic_ulcer_disease
0.0    1961
1.0      45
NaN       2
Name: count, dtype: int64

diabetes
diabetes
0    1542
1     466
Name: count, dtype: int64

mod_severe_ckd
mod_severe_ckd
0.0    1532
1.0     474
NaN       2
Name: count, dtype: int64

hemiplegia
hemiplegia
0    1996
1      12
Name: count, dtype: int64

leukemia
leukemia
0    2008
Name: count, dtype: int64

malignant_lymphoma
malignant_lymphoma
0    2007
1       1
Name: count, dtype: int64

solid_tumor
solid_tumor
0    1969
1      39
Name: count, dtype: int64

liver_disease
liver_disease
0.0    1923
1.0      84
NaN       1
Name: count, dtype: int64

aids
aids
0    2004
1       4
Name: count, dtype:

### ***4. Consistent text labels prevent the same category from appearing under different names in charts***

In [121]:
dfPH["respiratory_failure_type2"] = (
    dfPH["respiratory_failure_type2"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print(dfPH["respiratory_failure_type2"].value_counts(dropna=False))

respiratory_failure_type2
nontypeii    1894
typeii        114
Name: count, dtype: Int64


### ***5. cci_score can be used to compare groups of patients, so check its range and missing values. Do not invent scores for patients whose values are missing***

In [122]:
dfPH["cci_score"] = pd.to_numeric(
    dfPH["cci_score"],
    errors="coerce"
).astype("Int64")

print(dfPH["cci_score"].value_counts(dropna=False).sort_index())
print("Missing CCI scores:", dfPH["cci_score"].isna().sum())
dfPH["cci_score"].describe()

cci_score
0        56
1       770
2       699
3       368
4        94
5        15
6         1
<NA>      5
Name: count, dtype: Int64
Missing CCI scores: 5


count      2003.0
mean     1.861707
std      0.961469
min           0.0
25%           1.0
50%           2.0
75%           2.0
max           6.0
Name: cci_score, dtype: Float64

### ***6. Checking the cleaned file and saving the file. To confirm that cleaning did not accidentally remove patients or create duplicate IDs before joining this file to the other datasets***

In [123]:
print("Original rows:", len(df_original))
print("Cleaned rows:", len(dfPH))
print("Unique patient IDs:", dfPH["patient_id"].nunique())
print("Duplicate patient IDs:", dfPH["patient_id"].duplicated().sum())

print("\nRemaining missing values:")
display(dfPH.isna().sum())

assert len(dfPH) == 2008
assert dfPH["patient_id"].is_unique


dfPH.to_csv(
    "cleaned_files/patienthistory_clean.csv",
    index=False
)

print("Saved patienthistory_clean.csv")

Original rows: 2008
Cleaned rows: 2008
Unique patient IDs: 2008
Duplicate patient IDs: 0

Remaining missing values:


patient_id                   0
cerebrovascular_disease      0
dementia                     0
copd                         0
connective_tissue_disease    0
peptic_ulcer_disease         2
diabetes                     0
mod_severe_ckd               2
hemiplegia                   0
leukemia                     0
malignant_lymphoma           0
solid_tumor                  0
liver_disease                1
aids                         0
cci_score                    5
respiratory_failure_type2    0
acute_renal_failure          0
dtype: int64

Saved patienthistory_clean.csv


# ***Patient Prescriptions***

### **1. Read the CSV file and inspect the data. This confirms the file loaded correctly and shows missing values before you change anything**

In [124]:
import pandas as pd
import numpy as np

precriptions = pd.read_csv("data_files/patient_precriptions.csv")

# 1) Basic inspection
print(precriptions.shape)
print(precriptions.head())
print(precriptions.isna().sum())
print("Duplicate rows:", precriptions.duplicated().sum())
print("Duplicate patient-drug pairs:", precriptions.duplicated(subset=["inpatient_number", "drug_name"]).sum())


(15362, 2)
   inpatient_number                                         drug_name
0            857781                  sulfotanshinone sodium injection
1            857781                                 Furosemide tablet
2            857781                       Enoxaparin Sodium injection
3            857781  Meglumine Adenosine Cyclophosphate for injection
4            857781                              Furosemide injection
inpatient_number    0
drug_name           0
dtype: int64
Duplicate rows: 0
Duplicate patient-drug pairs: 0


### **2. Rename the patient ID to be consistent across multiple data files**

In [125]:
# 2) Clean patient ID
precriptions = precriptions.rename(columns={'inpatient_number': 'patient_id'})

### **3. Standardize the drug_name column values - removing unnecessary white spaces and lower the words to be in consistent with other values.**

In [126]:
# 3) Clean drug names
precriptions["drug_name"] = precriptions["drug_name"].astype(str).str.strip()
precriptions["drug_name"] = precriptions["drug_name"].str.replace(r"\s+", " ", regex=True)
precriptions["drug_name"] = precriptions["drug_name"].str.lower()

### **4. Check for any missing drug entries or with blank names. These records can be removed as they does not add any value to the dataset when performing descriptive or predective analysis to the model.**

In [127]:
# 4) Remove blank or missing drug entries
precriptions = precriptions[precriptions["drug_name"].notna() & precriptions["drug_name"].str.strip().ne("")]


### **5. Save the cleaned prescriptions data file**

In [128]:
precriptions.to_csv("cleaned_files/prescriptions_clean.csv", index=False)
print("Saved prescriptions_clean.csv")

Saved prescriptions_clean.csv


# ***Labs***

### **1. Load and inspect the file**


In [129]:
import pandas as pd
import numpy as np
import nbformat 

labs = pd.read_csv("data_files/labs.csv")

print(labs.shape)       # 2,008 rows, 107 columns
display(labs.head())
labs.info()


(2008, 107)


,inpatient_number,body_temperature,pulse,respiration,systolic_blood_pressure,diastolic_blood_pressure,map_value,fio2,creatinine_enzymatic_method,urea,...,measured_residual_base,measured_bicarbonate,carboxyhemoglobin,body_temperature_blood_gas,oxygen_saturation,partial_oxygen_pressure,oxyhemoglobin,anion_gap,free_calcium,total_hemoglobin
0,857781,36.7,87,19,102,64,76.666667,33,108.3,12.55,...,-2.1,21.2,0.4,37.0,97.0,93.0,95.9,17.8,1.14,125.0
1,743087,36.8,95,18,150,70,96.666667,33,62.0,4.29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,866418,36.5,98,18,102,67,78.666667,33,185.1,15.99,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,775928,36.0,73,19,110,74,86.000000,33,104.8,8.16,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,810128,35.0,88,19,134,62,86.000000,33,83.9,6.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Columns: 107 entries, inpatient_number to total_hemoglobin
dtypes: float64(101), int64(6)
memory usage: 1.6 MB


### **2. Check missing values and duplicates. This file has no duplicate rows or patient IDs. Many lab columns have missing values because a test was not recorded for every patient. Do not replace all those blanks with zero.**



In [130]:
missing = labs.isna().sum().sort_values(ascending=False)
print(missing)

print("Duplicate rows:", labs.duplicated().sum())
print("Duplicate patient IDs:", labs["inpatient_number"].duplicated().sum())


cholinesterase              2008
homocysteine                1862
apolipoprotein_a            1832
apolipoprotein_b            1832
lipoprotein                 1832
                            ... 
diastolic_blood_pressure       0
systolic_blood_pressure        0
respiration                    0
pulse                          0
inpatient_number               0
Length: 107, dtype: int64
Duplicate rows: 0
Duplicate patient IDs: 0


### **3. Remove the completely empty column. The Cholinesterase column is completely empty and hence cannot be used for further analysis. So dropping the column from labs.**



In [131]:
empty_columns = labs.columns[labs.isna().all()]
print(empty_columns.tolist())  # ['cholinesterase']

labs = labs.drop(columns=empty_columns)


['cholinesterase']


### **4. Mark unusable vital signs as missing. Three rows have blood pressure recorded as zero; two have systolic pressure below diastolic pressure. Mark the blood pressure values and their calculated map_value as missing in those five rows.**



In [132]:
invalid_bp = (
    (labs["systolic_blood_pressure"] == 0) |
    (labs["diastolic_blood_pressure"] == 0) |
    (labs["systolic_blood_pressure"] < labs["diastolic_blood_pressure"])
)

print("Rows with invalid blood pressure:", invalid_bp.sum())

labs.loc[
    invalid_bp,
    ["systolic_blood_pressure", "diastolic_blood_pressure", "map_value"]
] = np.nan

labs.loc[labs["pulse"] == 0, "pulse"] = np.nan
labs.loc[labs["respiration"] == 0, "respiration"] = np.nan


Rows with invalid blood pressure: 5


In [133]:
keep_cols = [
    "inpatient_number",
    "body_temperature",
    "pulse",
    "respiration",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "fio2",
    "creatinine_enzymatic_method",
    "urea",
    "uric_acid",
    "glomerular_filtration_rate",
    "cystatin",
    "white_blood_cell",
    "monocyte_count",
    "red_blood_cell",
    "hematocrit",
    "lymphocyte_count",
    "hemoglobin",
    "platelet",
    "basophil_count",
    "eosinophil_count",
    "neutrophil_count",
    "activated_partial_thromboplastin_time",
    "fibrinogen",
    "high_sensitivity_troponin",
    "myoglobin",
    "calcium",
    "potassium",
    "chloride",
    "sodium",
    "serum_magnesium",
    "creatine_kinase",
    "lactate_dehydrogenase",
    "brain_natriuretic_peptide",
    "albumin",
    "globulin",
    "total_protein",
    "cholesterol",
    "triglyceride",
    "glucose_blood_gas",
    "lactate",
    "partial_oxygen_pressure",
    "ph",
    "potassium_ion",
    "chloride_ion",
    "sodium_ion",
    "free_calcium"
]



### **5. Drop these derived / redundant columns**

***1. Derived values should not dominate the dataset***

Columns like:

map_value,
mean_corpuscular_volume,
mean_hemoglobin_volume,
mean_hemoglobin_concentration,
standard_bicarbonate,
anion_gap. 
are typically calculated from more basic variables. If you already have the raw values, these can be recreated later in analysis or feature engineering.

***2. Ratios and percentages are often redundant***

Variables such as:

monocyte_ratio,
basophil_ratio,
eosinophil_ratio,
neutrophil_ratio,
white_globulin_ratio  
are often informative, but they are not always necessary when the absolute count variables are already present. A model can always derive ratios later if needed.

***3. Blood-gas “derived” acid-base variables are not primary features***

These variables are often used for clinical interpretation, but for a simplified modeling dataset they are less useful than the direct measures:

pH,
partial_oxygen_pressure,
partial_pressure_of_carbon_dioxide,
lactate,
glucose_blood_gas 

So keeping the direct values and dropping the computed acid-base summaries is a sensible choice.

***4. Some columns are duplicated in meaning***

Examples:

total_protein and globulin are both part of protein balance,
mean_platelet_volume and platelet are different conceptually, but the calculated index is often less valuable for a reduced dataset,
international_normalized_ratio and prothrombin_time_ratio are related to the same coagulation process.
These are good candidates for dropping when working with a smaller variable set.

***5. A smaller dataset is easier to model and interpret***

Your goal is not to keep every possible field, but to keep:

stable raw measurements,
clinically important markers,
variables that are interpretable without complex calculations,
This makes the dataset easier to clean, visualize, and model without creating noise from highly derived features


In [134]:
drop_cols = [
    "map_value",
    "monocyte_ratio",
    "basophil_ratio",
    "eosinophil_ratio",
    "neutrophil_ratio",
    "mean_corpuscular_volume",
    "mean_hemoglobin_volume",
    "mean_hemoglobin_concentration",
    "mean_platelet_volume",
    "platelet_distribution_width",
    "platelet_hematocrit",
    "coefficient_of_variation_of_red_blood_cell_distribution_width",
    "standard_deviation_of_red_blood_cell_distribution_width",
    "international_normalized_ratio",
    "prothrombin_time_ratio",
    "prothrombin_activity",
    "white_globulin_ratio",
    "standard_residual_base",
    "measured_residual_base",
    "standard_bicarbonate",
    "measured_bicarbonate",
    "total_carbon_dioxide",
    "anion_gap",
    "oxygen_saturation",
    "oxyhemoglobin",
    "carboxyhemoglobin",
    "methemoglobin",
    "hematocrit_blood_gas",
    "total_hemoglobin",
    "hydroxybutyrate_dehydrogenase_to_lactate_dehydrogenase",
    "creatine_kinase_isoenzyme_to_creatine_kinase",
    "total_bile_acid",
    "total_protein",
    "globulin"
]

In [135]:
labs = labs.drop(columns=drop_cols, errors="ignore")

### **6. Rename the columns for the reduced labs dataframe. Keep the names short, consistent, and clinically readable. This helps later modeling and makes the notebook easier to follow. For example:**

**Short and readable:** systolic_bp, diastolic_bp, gfr, wbc, po2\
**Consistent format:** all names are lowercase snake_case\
**Medical convention:** aptt, bnp, ck, ldh, hs_troponin\
**Clearer for analysis:** patient_id instead of inpatient_number




In [136]:
rename_map = {
    "inpatient_number": "patient_id",
    "body_temperature": "body_temp",
    "pulse": "pulse",
    "respiration": "respiration",
    "systolic_blood_pressure": "systolic_bp",
    "diastolic_blood_pressure": "diastolic_bp",
    "fio2": "fio2",
    "creatinine_enzymatic_method": "creatinine",
    "urea": "urea",
    "uric_acid": "uric_acid",
    "glomerular_filtration_rate": "gfr",
    "cystatin": "cystatin",
    "white_blood_cell": "wbc",
    "monocyte_count": "monocyte_count",
    "red_blood_cell": "rbc",
    "hematocrit": "hematocrit",
    "lymphocyte_count": "lymphocyte_count",
    "hemoglobin": "hemoglobin",
    "platelet": "platelet_count",
    "basophil_count": "basophil_count",
    "eosinophil_count": "eosinophil_count",
    "neutrophil_count": "neutrophil_count",
    "activated_partial_thromboplastin_time": "aptt",
    "fibrinogen": "fibrinogen",
    "high_sensitivity_troponin": "hs_troponin",
    "myoglobin": "myoglobin",
    "calcium": "calcium",
    "potassium": "potassium",
    "chloride": "chloride",
    "sodium": "sodium",
    "serum_magnesium": "magnesium",
    "creatine_kinase": "ck",
    "lactate_dehydrogenase": "ldh",
    "brain_natriuretic_peptide": "bnp",
    "albumin": "albumin",
    "globulin": "globulin",
    "total_protein": "total_protein",
    "cholesterol": "cholesterol",
    "triglyceride": "triglyceride",
    "glucose_blood_gas": "glucose_blood_gas",
    "lactate": "lactate",
    "partial_oxygen_pressure": "po2",
    "ph": "ph",
    "potassium_ion": "potassium_ion",
    "chloride_ion": "chloride_ion",
    "sodium_ion": "sodium_ion",
    "free_calcium": "free_calcium",
    "partial_pressure_of_carbon_dioxide": "paco2"
}

labs = labs.rename(columns=rename_map)

In [137]:
labs.describe()
labs.isna().sum()
display(labs.dtypes)

patient_id                      int64
body_temp                     float64
pulse                         float64
respiration                   float64
systolic_bp                   float64
                               ...   
glucose_blood_gas             float64
lactate                       float64
body_temperature_blood_gas    float64
po2                           float64
free_calcium                  float64
Length: 72, dtype: object

### **7. Flag impossible or clinically unrealistic values. This catches data-entry issues and impossible physiology values. It is often more useful than blindly dropping rows.**
There are likely other impossible values across the dataset.

Examples:

temperature < 30 or > 45\
pulse < 20 or > 200\
respiration < 4 or > 60\
sodium < 100 or > 180\
potassium < 1 or > 8\
creatinine < 0

In [138]:
for col, low, high in [
    ("body_temp", 30, 45),
    ("pulse", 20, 200),
    ("respiration", 4, 60),
    ("sodium", 100, 180),
    ("potassium", 1, 8),
    ("creatinine", 0, 20)
]:
    if col in labs.columns:
        labs.loc[(labs[col] < low) | (labs[col] > high), col] = np.nan

### **8. Check and standardize column types. Some lab values may still be read as strings, especially if there are commas, units, or blanks.Numeric columns should be numeric for analysis. This catches values like "12.5" and "NA" that are not properly converted.**

In [139]:
numeric_cols = [
    "body_temp",
    "pulse",
    "respiration",
    "systolic_bp",
    "diastolic_bp",
    "fio2",
    "creatinine",
    "urea",
    "uric_acid",
    "gfr",
    "cystatin",
    "wbc",
    "monocyte_count",
    "rbc",
    "hematocrit",
    "lymphocyte_count",
    "hemoglobin",
    "platelet_count",
    "basophil_count",
    "eosinophil_count",
    "neutrophil_count",
    "aptt",
    "fibrinogen",
    "hs_troponin",
    "myoglobin",
    "calcium",
    "potassium",
    "chloride",
    "sodium",
    "magnesium",
    "ck",
    "ldh",
    "bnp",
    "albumin",
    "globulin",
    "total_protein",
    "cholesterol",
    "triglyceride",
    "glucose_blood_gas",
    "lactate",
    "po2",
    "ph",
    "potassium_ion",
    "chloride_ion",
    "sodium_ion",
    "free_calcium"
]

for col in numeric_cols:
    if col in labs.columns:
        labs[col] = pd.to_numeric(labs[col], errors="coerce")

In [140]:
display(labs.head())

,patient_id,body_temp,pulse,respiration,systolic_bp,diastolic_bp,fio2,creatinine,urea,uric_acid,...,paco2,reduced_hemoglobin,potassium_ion,chloride_ion,sodium_ion,glucose_blood_gas,lactate,body_temperature_blood_gas,po2,free_calcium
0,857781,36.7,87.0,19.0,102.0,64.0,33,NaN,12.55,685.0,...,32.0,3.4,5.63,103.0,136.4,5.8,2.5,37.0,93.0,1.14
1,743087,36.8,95.0,18.0,150.0,70.0,33,NaN,4.29,170.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,866418,36.5,98.0,18.0,102.0,67.0,33,NaN,15.99,567.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,775928,36.0,73.0,19.0,110.0,74.0,33,NaN,8.16,635.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,810128,35.0,88.0,19.0,134.0,62.0,33,NaN,6.86,432.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [141]:
labs.shape

(2008, 72)

In [142]:
labs.to_csv("cleaned_files/labs_clean.csv", index=False)
print("Saved labs_clean.csv")

Saved labs_clean.csv


# ***Hospitalization Discharge***

### **1. Read and inspect the hospitalization discharge file**

In [143]:
hd=pd.read_csv("data_files/hospitalization_discharge.csv")

print(hd.shape)
print(hd.head())

(2008, 21)
   inpatient_number destinationdischarge admission_ward admission_way  \
0            857781                 Home     Cardiology  NonEmergency   
1            743087                 Home     Cardiology  NonEmergency   
2            866418                 Home     Cardiology  NonEmergency   
3            775928                 Home     Cardiology     Emergency   
4            810128                 Home     Cardiology  NonEmergency   

  discharge_department  visit_times respiratory_support oxygen_inhalation  \
0           Cardiology            1                 NaN     OxygenTherapy   
1           Cardiology            1                 NaN     OxygenTherapy   
2           Cardiology            2                 NaN     OxygenTherapy   
3           Cardiology            1                 NaN     OxygenTherapy   
4           Cardiology            1                 NaN     OxygenTherapy   

   dischargeday       admission_date  ... death_within_28_days  \
0            11  2017

### **2. Check for missing values and duplicates to avoid data redundancy**

In [144]:
hd.info()
print(hd.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 21 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   inpatient_number                                2008 non-null   int64  
 1   destinationdischarge                            2008 non-null   object 
 2   admission_ward                                  2008 non-null   object 
 3   admission_way                                   2008 non-null   object 
 4   discharge_department                            2008 non-null   object 
 5   visit_times                                     2008 non-null   int64  
 6   respiratory_support                             42 non-null     object 
 7   oxygen_inhalation                               2008 non-null   object 
 8   dischargeday                                    2008 non-null   int64  
 9   admission_date                           

### **3. Remove completely empty columns as they cannot be further used for analysis**

In [145]:
empty_columns = hd.columns[hd.isna().all()]
print(empty_columns.tolist()) 

hd = hd.drop(columns=empty_columns)


[]


### **4. Check for column datatyes and change the columns to appropriate datatypes('admission_date' from string to datatime) for data consistency and accurate filtering for better satistical analysis**

In [146]:
hd.dtypes
hd['admission_date'] = pd.to_datetime(hd['admission_date'], errors='coerce')



### **5. Rename columns for better readability and consistency**


In [147]:
hd.rename(columns={
    "inpatient_number": "patient_id",
    "destinationdischarge": "discharge_destination",
    "admission_ward": "admission_ward",
    "admission_way": "admission_mode",
    "discharge_department": "discharge_department",
    "visit_times": "visit_count",
    "respiratory_support": "resp_support",
    "oxygen_inhalation": "oxygen_use",
    "dischargeday": "discharge_day",
    "admission_date": "admission_date",
    "outcome_during_hospitalization": "in_hosp_outcome",
    "death_within_28_days": "mortality_28d",
    "re_admission_within_28_days": "readmission_28d",
    "death_within_3_months": "death_3mo",
    "re_admission_within_3_months": "readmission_3mo",
    "death_within_6_months": "mortality_3mo",
    "re_admission_within_6_months": "readmission_6mo",
    "time_of_death__days_from_admission": "death_time_days",
    "readmission_time_days_from_admission": "readmission_time_days",
    "return_to_emergency_department_within_6_months": "ed_return_6mo",
    "time_to_emergency_department_within_6_months": "ed_return_time_days"
}, inplace=True)

### **6. Verify changes and load clean data**

In [148]:
hd.info()
hd.to_csv("cleaned_files/hospitalization_discharge_clean.csv", index=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   patient_id             2008 non-null   int64         
 1   discharge_destination  2008 non-null   object        
 2   admission_ward         2008 non-null   object        
 3   admission_mode         2008 non-null   object        
 4   discharge_department   2008 non-null   object        
 5   visit_count            2008 non-null   int64         
 6   resp_support           42 non-null     object        
 7   oxygen_use             2008 non-null   object        
 8   discharge_day          2008 non-null   int64         
 9   admission_date         2008 non-null   datetime64[ns]
 10  in_hosp_outcome        2008 non-null   object        
 11  mortality_28d          2008 non-null   int64         
 12  readmission_28d        2008 non-null   int64         
 13  dea

# ***Responsivenss***

### **1. Read and inspect responsivenes file**

In [149]:
resp=pd.read_csv("data_files/responsivenes.csv")

print(resp.shape)
print(resp.head())

(2008, 6)
   inpatient_number  eye_opening  verbal_response  movement consciousness  gcs
0            857781            4                5         6         Clear   15
1            743087            4                5         6         Clear   15
2            866418            4                5         6         Clear   15
3            775928            4                5         6         Clear   15
4            810128            4                5         6         Clear   15


### **2. Check missing values and duplicates to avoid data redundancy**

In [150]:
print(resp.isna().sum())
print(resp.duplicated().sum())

inpatient_number    0
eye_opening         0
verbal_response     0
movement            0
consciousness       0
gcs                 0
dtype: int64
0


### **3. Remove completely empty columns if any as they cannot be used for analysis**

In [151]:
empty_columns = resp.columns[resp.isna().all()]
print(empty_columns.tolist())

[]


### **4. Check for column datatypes and change if not appropriate**

resp.dtypes

### **5. Rename "inpatient_number" to "patient_id" for data consistency and better readability**

In [152]:
resp.rename(columns={
    "inpatient_number": "patient_id"}, inplace=True)

### **6.Verify changes and load clean data**

In [153]:
resp.info()
resp.to_csv("cleaned_files/responsivenes_clean.csv", index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   patient_id       2008 non-null   int64 
 1   eye_opening      2008 non-null   int64 
 2   verbal_response  2008 non-null   int64 
 3   movement         2008 non-null   int64 
 4   consciousness    2008 non-null   object
 5   gcs              2008 non-null   int64 
dtypes: int64(5), object(1)
memory usage: 94.2+ KB


In [ ]:
agg_prescreption = pd.read_csv("cleaned_files/prescriptions_clean.csv")

agg_prescreption = agg_prescreption.rename(columns={"inpatient_number": "patient_id"})
agg_prescreption["drug_name"] = agg_prescreption["drug_name"].astype(str).str.strip().str.lower()

patient_drug_summary = (
    agg_prescreption.groupby("patient_id")
      .agg(
          total_drugs=("drug_name", "count"),
          drugs_list=("drug_name", lambda x: ", ".join(sorted(set(x))))
      )
      .reset_index()
)

print(patient_drug_summary.head())

   patient_id  total_drugs                                         drugs_list
0      722128            7  deslanoside injection, digoxin tablet, furosem...
1      723327           12  aspirin enteric-coated tablet, atorvastatin ca...
2      723617            4  atorvastatin calcium tablet, furosemide inject...
3      724385            6  deslanoside injection, digoxin tablet, furosem...
4      725509            9  deslanoside injection, digoxin tablet, furosem...


In [155]:
patient_drug_summary.to_csv("cleaned_files/summarized_prescreption_clean.csv", index=False)

## **Cardiac Complications Cleanig**

### **1. Read the Csv File and inspect the data**


In [27]:
dfcardiac = pd.read_csv("data_files/cardiac_complications.csv")

# Standardize column names for clean analysis
# Keep names lowercase and replace spaces/symbols with underscores

dfcardiac.columns = dfcardiac.columns.str.strip().str.lower()
dfcardiac.columns = dfcardiac.columns.str.replace(r"[^a-z0-9]+", "_", regex=True)

dfcardiac = dfcardiac.rename(columns={
    "inpatient_number": "patient_id",
    "type_of_heart_failure": "heart_failure_type",
    "left_ventricular_end_diastolic_diameter_lv": "lv_end_diastolic_diameter"
})

print(dfcardiac.shape)
print(dfcardiac.info())
print(dfcardiac.columns.tolist())


(2008, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 14 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   patient_id                            2008 non-null   int64  
 1   nyha_cardiac_function_classification  2008 non-null   int64  
 2   killip_grade                          2008 non-null   int64  
 3   myocardial_infarction                 2008 non-null   int64  
 4   congestive_heart_failure              2008 non-null   int64  
 5   peripheral_vascular_disease           2008 non-null   int64  
 6   heart_failure_type                    2008 non-null   object 
 7   lvef                                  635 non-null    float64
 8   lv_end_diastolic_diameter             1311 non-null   float64
 9   mitral_valve_ems                      980 non-null    float64
 10  mitral_valve_ams                      550 non-null    float64
 11  ea    

### **2. Renamed some column names to meaningful names.**

In [28]:
dfcardiac = dfcardiac.rename(columns={
    "inpatient_number": "patient_id",
    "type_of_heart_failure": "heart_failure_type",
    "left_ventricular_end_diastolic_diameter_lv": "lv_end_diastolic_diameter"
})

print(dfcardiac.shape)
print(dfcardiac.info())
print(dfcardiac.columns.tolist())

(2008, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 14 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   patient_id                            2008 non-null   int64  
 1   nyha_cardiac_function_classification  2008 non-null   int64  
 2   killip_grade                          2008 non-null   int64  
 3   myocardial_infarction                 2008 non-null   int64  
 4   congestive_heart_failure              2008 non-null   int64  
 5   peripheral_vascular_disease           2008 non-null   int64  
 6   heart_failure_type                    2008 non-null   object 
 7   lvef                                  635 non-null    float64
 8   lv_end_diastolic_diameter             1311 non-null   float64
 9   mitral_valve_ems                      980 non-null    float64
 10  mitral_valve_ams                      550 non-null    float64
 11  ea    

### **3. Standardize all column names with lower-case and underscores.**


In [29]:
dfcardiac.columns = dfcardiac.columns.str.strip().str.lower()
dfcardiac.columns = dfcardiac.columns.str.replace(r"[^a-z0-9]+", "_", regex=True)

### **4.Remove duplicate rows.**

In [30]:
dfcardiac = dfcardiac.drop_duplicates()

### **5. Check missing values.**

In [31]:
dfcardiac.isna().sum().sort_values(ascending=False)


tricuspid_valve_return_pressure         1826
ea                                      1615
mitral_valve_ams                        1458
lvef                                    1373
tricuspid_valve_return_velocity         1218
mitral_valve_ems                        1028
lv_end_diastolic_diameter                697
patient_id                                 0
nyha_cardiac_function_classification       0
killip_grade                               0
myocardial_infarction                      0
congestive_heart_failure                   0
peripheral_vascular_disease                0
heart_failure_type                         0
dtype: int64

### **6. Handle missing values based on type.**

In [32]:
dfcardiac = dfcardiac.fillna(dfcardiac.median(numeric_only=True))

### **7. Convert object/string columns to proper format.**

In [33]:
dfcardiac["heart_failure_type"] = (
    dfcardiac["heart_failure_type"]
    .astype(str)
    .str.strip()
)

### **8. Review and clean categorical values.**

In [38]:
dfcardiac["heart_failure_type"] = dfcardiac["heart_failure_type"].str.lower().str.strip()
##If values are inconsistent, normalize them:#
dfcardiac["heart_failure_type"] = dfcardiac["heart_failure_type"].str.lower().str.strip()

### **9. Check for impossible or invalid numeric values.**

In [35]:
for col in dfcardiac.select_dtypes(include=["number"]).columns:
    print(col, dfcardiac[col].describe())

patient_id count      2008.000000
mean     797747.542829
std       41127.801740
min      722128.000000
25%      763164.500000
50%      798758.000000
75%      829399.750000
max      905720.000000
Name: patient_id, dtype: float64
nyha_cardiac_function_classification count    2008.000000
mean        3.130976
std         0.682383
min         2.000000
25%         3.000000
50%         3.000000
75%         4.000000
max         4.000000
Name: nyha_cardiac_function_classification, dtype: float64
killip_grade count    2008.000000
mean        1.992530
std         0.759884
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         4.000000
Name: killip_grade, dtype: float64
myocardial_infarction count    2008.000000
mean        0.071215
std         0.257248
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: myocardial_infarction, dtype: float64
congestive_heart_failure count    2008.000000
mean        

### **10. Final dataset validation.**

In [36]:
dfcardiac.shape
dfcardiac.isna().sum().sum()
dfcardiac.dtypes

patient_id                                int64
nyha_cardiac_function_classification      int64
killip_grade                              int64
myocardial_infarction                     int64
congestive_heart_failure                  int64
peripheral_vascular_disease               int64
heart_failure_type                       object
lvef                                    float64
lv_end_diastolic_diameter               float64
mitral_valve_ems                        float64
mitral_valve_ams                        float64
ea                                      float64
tricuspid_valve_return_velocity         float64
tricuspid_valve_return_pressure         float64
dtype: object

In [37]:

dfcardiac.to_csv("cleaned_files/cardiac_cleaned.csv", index=False)



